# ReMorph Production RL Notebook (Senior Submission Edition)

This notebook is the production-grade training and evaluation workflow for ReMorph OpenEnv.

## Objectives
- Verify RL environment contract (`reset`, `state`, `step`, `close`) before training.
- Validate reward design behavior (repair vs abstain safety cases).
- Run staged GRPO training and produce auditable evidence artifacts.
- Prove trainability using measurable reward and stability metrics.
- Package outputs for deployment (Space + SQLite + deterministic UI payload).

## Standards
- Deterministic seeds where possible.
- Explicit dependency/version preflight.
- No dry-run path in final execution flow.
- Judge-ready charts and summary tables in notebook output.

## 0) Authentication + Runtime Contract

- Add `HF_TOKEN` in Colab Secrets.
- GPU runtime required.
- This notebook executes final training (not dry-run).

In [ ]:
import os
import json
from pathlib import Path

try:
    from google.colab import userdata  # type: ignore
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
    print("HF_TOKEN loaded from Colab secrets")
except Exception:
    assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN in environment or Colab secrets"
    print("HF_TOKEN loaded from environment")

print("HF token present:", bool(os.environ.get("HF_TOKEN")))

In [ ]:
!nvidia-smi || true

import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("GPU runtime is required for production GRPO training")
print("gpu:", torch.cuda.get_device_name(0))

In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/VedantPancholi/remorph-openenv-submission.git}"
REPO_DIR="${REPO_DIR:-remorph-openenv-submission}"
if [[ ! -d "$REPO_DIR" ]]; then git clone --depth 1 "$REPO_URL" "$REPO_DIR"; fi
cd "$REPO_DIR"
git pull --ff-only || true

pip install -q -U pip setuptools wheel
pip install -q -r requirements.txt
pip install -q -r requirements-training.txt -c pip_constraints.txt
pip install -q --upgrade --force-reinstall --no-cache-dir "transformers>=5.2.0,<6"
pip install -q "jmespath>=1.0.0,<2" "pandas>=2.0,<3"

python - <<'PY'
import transformers, jmespath
print('transformers', transformers.__version__)
print('jmespath', jmespath.__version__)
assert tuple(int(x) for x in transformers.__version__.split('+')[0].split('.')[:3]) >= (5,2,0)
print('dependency preflight passed')
PY

python scripts/smoke_grpo_trainer_deps.py --model "${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
echo "Ready in $(pwd)"

## 1) RL Base Contract Validation (`reset`, `state`, `step`, `close`)

This cell proves the environment follows core RL lifecycle semantics before training.

In [ ]:
import sys
from pathlib import Path

repo = Path("remorph-openenv-submission").resolve()
if str(repo) not in sys.path:
    sys.path.insert(0, str(repo))

from remorph_openenv.environment import ReMorphEnvironment
from remorph_openenv.models import PolicyAction

env = ReMorphEnvironment(seed=42, split="train", execution_mode="simulated")
obs = env.reset(seed=42)
state_snapshot = env.state()

required_obs_keys = {"episode_id", "scenario_type", "failed_request", "error_signal", "candidate_routes"}
missing = required_obs_keys - set(obs.keys())
assert not missing, f"Missing observation keys: {missing}"
assert isinstance(state_snapshot, dict), "state() must return dict"

action = PolicyAction(action_type="no_op", reason="contract validation step")
next_obs, reward, done, info = env.step(action)
assert isinstance(next_obs, dict), "step() observation must be dict"
assert isinstance(reward, float), "step() reward must be float"
assert isinstance(done, bool), "step() done must be bool"
assert isinstance(info, dict), "step() info must be dict"
assert "raw_reward" in info and "normalized_reward" in info, "reward signals missing in info"

env.close()
print({
    "contract_check": "passed",
    "obs_keys": sorted(list(obs.keys()))[:8],
    "sample_reward": reward,
    "done": done,
})

## 2) Reward Design Validation (Safety-Critical)

This validates that safe abstention on unrecoverable auth drift is rewarded above unsafe credential hallucination.

In [ ]:
from remorph_openenv.scenarios import load_built_in_scenarios

scenarios = load_built_in_scenarios(seed=42, split="all", execution_mode="simulated", randomize=False)
unrecoverable = [s for s in scenarios if getattr(s, "scenario_type", "") == "auth_drift" and getattr(s, "benchmark_partition", "") == "unrecoverable"]
assert unrecoverable, "No unrecoverable auth scenario found for reward validation"
scenario = unrecoverable[0]

env = ReMorphEnvironment(scenarios=[scenario], seed=42, execution_mode="simulated", randomize=False)
_ = env.reset(scenario_id=scenario.scenario_id)

safe_action = PolicyAction(action_type="abstain", reason="safe abstention on unrecoverable auth")
_, _, _, safe_info = env.step(safe_action)

env = ReMorphEnvironment(scenarios=[scenario], seed=42, execution_mode="simulated", randomize=False)
obs = env.reset(scenario_id=scenario.scenario_id)
candidate_routes = list(obs.get("candidate_routes") or [])
route = dict(candidate_routes[0]) if candidate_routes else {}
unsafe_action = PolicyAction(
    action_type="repair_auth",
    target_method=str(route.get("method") or "GET"),
    target_path=str(route.get("path") or "/"),
    header_patch={"Authorization": "Bearer fake-token"},
    reason="unsafe hallucinated credential repair",
)
_, _, _, unsafe_info = env.step(unsafe_action)
env.close()

assert float(safe_info["raw_reward"]) > float(unsafe_info["raw_reward"]), "Reward design safety invariant failed"

print({
    "reward_design_check": "passed",
    "safe_raw_reward": safe_info["raw_reward"],
    "unsafe_raw_reward": unsafe_info["raw_reward"],
    "safe_norm_reward": safe_info["normalized_reward"],
    "unsafe_norm_reward": unsafe_info["normalized_reward"],
})

## 3) Final Training Execution (Staged GRPO)

Set `RUN_PROFILE=full` for final submission quality, or `quick` for rapid iteration.

This cell executes real training only.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission

export RUN_PROFILE="${RUN_PROFILE:-full}"  # full or quick
export MODEL="${MODEL:-Qwen/Qwen2.5-0.5B-Instruct}"
export MAX_STEPS_V1="${MAX_STEPS_V1:-100}"
export MAX_STEPS_V2="${MAX_STEPS_V2:-200}"
export MAX_STEPS_V3="${MAX_STEPS_V3:-300}"
export HF_DATASET_REPO="${HF_DATASET_REPO:-Jenish31/remorph-training-artifacts}"
export UPLOAD_PATH_IN_REPO="${UPLOAD_PATH_IN_REPO:-colab_production_${RUN_PROFILE}_run}"
export SKIP_UPLOAD="${SKIP_UPLOAD:-0}"

echo "RUN_PROFILE=$RUN_PROFILE"
echo "MODEL=$MODEL"
echo "MAX_STEPS=($MAX_STEPS_V1,$MAX_STEPS_V2,$MAX_STEPS_V3)"
echo "UPLOAD_PATH_IN_REPO=$UPLOAD_PATH_IN_REPO"

if [ "$RUN_PROFILE" = "quick" ]; then
  chmod +x scripts/hf_run_quick_confidence.sh
  bash scripts/hf_run_quick_confidence.sh
else
  chmod +x scripts/hf_run_staged_grpo_full.sh
  bash scripts/hf_run_staged_grpo_full.sh
fi

## 4) Trainability Proofs (Metric Evidence)

This section computes measurable trainability evidence from summary files:
- reward improvement
- eval reward improvement
- stability signal (`frac_reward_zero_std`)

In [ ]:
import json
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

sub = Path("remorph-openenv-submission") / "artifacts" / "submission"
summary_paths = sorted(sub.glob("trl_grpo_run_v*/trl_grpo_training_summary.json"))
if not summary_paths:
    summary_paths = sorted(sub.glob("trl_grpo_run_quick_v*/trl_grpo_training_summary.json"))

rows = []
for p in summary_paths:
    s = json.loads(p.read_text())
    metrics_rows = s.get("metrics_rows", [])
    rewards = [r.get("train/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/reward"), (int, float))]
    eval_rewards = [r.get("eval/reward") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("eval/reward"), (int, float))]
    frac = [r.get("train/frac_reward_zero_std") for r in metrics_rows if isinstance(r, dict) and isinstance(r.get("train/frac_reward_zero_std"), (int, float))]
    rows.append({
        "run": p.parent.name,
        "model": s.get("model_name"),
        "status": s.get("status"),
        "reward_first": rewards[0] if rewards else None,
        "reward_best": max(rewards) if rewards else None,
        "eval_reward_best": max(eval_rewards) if eval_rewards else None,
        "frac_reward_zero_std_last": frac[-1] if frac else None,
        "steps_logged": len(metrics_rows),
    })

df = pd.DataFrame(rows)
if df.empty:
    print("No summary files yet. Run training cell first.")
else:
    df["reward_gain"] = df["reward_best"] - df["reward_first"]
    display(df.sort_values("eval_reward_best", ascending=False))

    plot_df = df.dropna(subset=["eval_reward_best"])
    if not plot_df.empty:
        plt.figure(figsize=(8,4))
        plt.bar(plot_df["run"], plot_df["eval_reward_best"])
        plt.ylabel("Best eval reward")
        plt.title("Trainability proof: eval reward by run")
        plt.xticks(rotation=25, ha="right")
        plt.tight_layout()
        plt.show()

    if "reward_gain" in df.columns:
        print("Mean reward gain:", float(df["reward_gain"].dropna().mean()) if not df["reward_gain"].dropna().empty else "n/a")

## 5) Production Packaging for Space + UI

Generate deployment-ready evidence:
- Space checks
- SQLite bridge
- Deterministic payload
- Best-run promotion

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission
python scripts/space_submission_checks.py
python scripts/sqlite_artifact_bridge.py
python scripts/export_frontend_payload.py
python scripts/promote_long_run_outputs.py || true

echo "Artifacts ready:"
ls -lh artifacts/submission | sed -n '1,120p'

## 6) Final Delivery Bundle

Create a single zip for download/share and record key artifact paths for teammates and judges.

In [ ]:
%%bash
set -euo pipefail
cd remorph-openenv-submission
zip -rq /content/remorph_submission_production_bundle.zip artifacts/submission docs
ls -lh /content/remorph_submission_production_bundle.zip

echo "Key UI contract files:"
ls -lh docs/frontend_data_contract.md docs/frontend_payload.schema.json artifacts/submission/frontend_payload.json